In [9]:
!pip install -q anthropic


In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# CELL 0 — CLAUDE CLIENT FROM COLAB SECRETS
# ============================================================
from anthropic import Anthropic

CLAUDE_API_KEY = ""

client = Anthropic(
    api_key=CLAUDE_API_KEY
)

print("Claude client initialized.")


Claude client initialized.


In [12]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================
import os
import json
import re
import time
import pandas as pd

CV_RUN = 1   # ONLY change this to 2,3,4,5 for the other folds

DATA_PATH = "/content/drive/MyDrive/conversation_en.json"
SPLIT_DIR = "/content/drive/MyDrive/5class_5cv_frozen"

TRAIN_IDS_PATH = f"{SPLIT_DIR}/cv{CV_RUN}_train_ids.csv"
VAL_IDS_PATH   = f"{SPLIT_DIR}/cv{CV_RUN}_val_ids.csv"
TEST_IDS_PATH  = f"{SPLIT_DIR}/cv{CV_RUN}_test_ids.csv"

OUTPUT_DIR = f"/content/drive/MyDrive/5class_5cv_augmentation/cv{CV_RUN}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Keep the original two-class augmentation targets unchanged.
CORE_TARGETS = {
    "Chitchat": 100,
    "Revise": 100,
}

# Inquire is generated separately because it is ultra-sparse.
INQUIRE_TARGET = 100

# Safety limits prevent an infinite generation loop.
CORE_MAX_API_CALLS_PER_CLASS = 250
INQUIRE_MAX_API_CALLS = 400

MODEL_NAME = "claude-sonnet-4-5"
TEMPERATURE = 0.4
MAX_TOKENS = 600

print("CV run:", CV_RUN)
print("Train IDs:", TRAIN_IDS_PATH)
print("Val IDs:", VAL_IDS_PATH)
print("Test IDs:", TEST_IDS_PATH)
print("Output directory:", OUTPUT_DIR)
print("Targets:", {**CORE_TARGETS, "Inquire": INQUIRE_TARGET})


CV run: 1
Train IDs: /content/drive/MyDrive/5class_5cv_frozen/cv1_train_ids.csv
Val IDs: /content/drive/MyDrive/5class_5cv_frozen/cv1_val_ids.csv
Test IDs: /content/drive/MyDrive/5class_5cv_frozen/cv1_test_ids.csv
Output directory: /content/drive/MyDrive/5class_5cv_augmentation/cv1
Targets: {'Chitchat': 100, 'Revise': 100, 'Inquire': 100}


In [13]:
# ============================================================
# CELL 2 — LOAD + VERIFY FROZEN CV IDS
# ============================================================
def load_id_set(path):
    df = pd.read_csv(path)
    assert "conversation_id" in df.columns, f"Missing conversation_id column in {path}"
    ids = df["conversation_id"].astype(str)
    assert ids.nunique() == len(ids), f"Duplicate IDs found in {path}"
    return set(ids)

train_cids = load_id_set(TRAIN_IDS_PATH)
val_cids   = load_id_set(VAL_IDS_PATH)
test_cids  = load_id_set(TEST_IDS_PATH)

expected_train = 528 if CV_RUN == 1 else 529
expected_val   = 132
expected_test  = 166 if CV_RUN == 1 else 165

print("Train conversations:", len(train_cids))
print("Validation conversations:", len(val_cids))
print("Test conversations:", len(test_cids))

assert len(train_cids) == expected_train
assert len(val_cids) == expected_val
assert len(test_cids) == expected_test
assert train_cids.isdisjoint(val_cids)
assert train_cids.isdisjoint(test_cids)
assert val_cids.isdisjoint(test_cids)
assert len(train_cids | val_cids | test_cids) == 826

print(f"PASS: CV{CV_RUN} frozen split loaded and conversation-disjoint.")


Train conversations: 528
Validation conversations: 132
Test conversations: 166
PASS: CV1 frozen split loaded and conversation-disjoint.


In [14]:
# ============================================================
# CELL 3 — LOAD ORIGINAL DATA; RECOVER EXACT TRAIN CONVERSATIONS
# ============================================================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

assert len(raw_data) == 826, f"Expected 826 English conversations, found {len(raw_data)}"

cv_train_conversations = [
    conv for conv in raw_data
    if str(conv["conv_id"]) in train_cids
]

recovered_train_cids = {str(conv["conv_id"]) for conv in cv_train_conversations}

assert len(cv_train_conversations) == expected_train
assert recovered_train_cids == train_cids

print(
    f"PASS: CV{CV_RUN} recovered exactly "
    f"{expected_train} frozen training conversations."
)


PASS: CV1 recovered exactly 528 frozen training conversations.


In [15]:
# ============================================================
# CELL 4 — BUILD THREE-RARE-CLASS SOURCE POOL
# ============================================================
AUGMENT_LABELS = ["Chitchat", "Revise", "Inquire"]
augmentation_sources = []

for conv in cv_train_conversations:
    conv_id = str(conv["conv_id"])
    conversation = conv["conversation"]

    for i, msg in enumerate(conversation):
        if msg.get("role") != "user":
            continue

        intent = msg.get("intent")
        if intent not in AUGMENT_LABELS:
            continue

        # Last system turn before this target user turn.
        previous_system_text = ""
        for previous_msg in reversed(conversation[:i]):
            if previous_msg.get("role") == "system":
                previous_system_text = previous_msg.get("content", "") or ""
                break

        augmentation_sources.append({
            "conversation_id": conv_id,
            "turn_index": int(i),
            "system_text": previous_system_text,
            "user_text": msg.get("content", "") or "",
            "label": intent,
        })

source_df = pd.DataFrame(augmentation_sources)
assert not source_df.empty

print(f"CV{CV_RUN} augmentation source distribution:")
print(source_df["label"].value_counts())

print("Source conversations by class:")
for label in AUGMENT_LABELS:
    subset = source_df[source_df["label"] == label]
    print(
        f"{label}: {len(subset)} source turns, "
        f"{subset['conversation_id'].nunique()} source conversations"
    )
    assert len(subset) > 0, f"No training source available for {label}"


CV1 augmentation source distribution:
label
Chitchat    173
Revise       39
Inquire      11
Name: count, dtype: int64
Source conversations by class:
Chitchat: 173 source turns, 132 source conversations
Revise: 39 source turns, 31 source conversations
Inquire: 11 source turns, 8 source conversations


In [16]:
# ============================================================
# CELL 5 — LABEL-PRESERVING AUGMENTATION PROMPT
# ============================================================
def build_augmentation_prompt(system_text, user_text, label):
    inquire_guard = """
SPECIAL RULES FOR INQUIRE:
- The user must remain explicitly information-seeking or clarification-seeking.
- Preserve the original question/request for information.
- Do NOT convert it into a preference, requirement, acceptance, rejection, or revision.
- Do NOT invent a new shopping preference.
- Use varied natural wording while keeping the same informational goal.
""" if label == "Inquire" else ""

    return f"""
You are an expert in e-commerce conversational AI and dialogue understanding.

Your task is to generate realistic, label-preserving variations of a conversation sample for intent classification training.

STRICT RULES:
- The intent label MUST remain EXACTLY the same.
- Do NOT change the user's goal or meaning.
- Do NOT introduce a different intent.
- Keep language natural, casual, and realistic.
- Do NOT make the text overly formal.
- Preserve conversational style, including short or imperfect phrasing when appropriate.
- If a variation changes the intent, do not generate it.
- Before returning each variation, verify internally that it still matches the provided label.
{inquire_guard}
INPUT
System: "{system_text}"
User: "{user_text}"
Intent Label: "{label}"

Generate exactly 3 variations:

Variation A - User-only
- Remove system message completely.
- Rewrite only the user message.
- Preserve enough information that the original intent remains clear without the system message.

Variation B - Natural Conversation
- Slightly paraphrase the system message.
- Slightly rewrite the user response.
- Keep semantics identical.

Variation C - Abstracted Context
- Replace system message with a short tag like [Clarify], [Recommend], [Confirm], or [Filter].
- Keep user response natural.

Return ONLY raw valid JSON in this exact format.
Do not use markdown.
Do not use code fences.

[
  {{
    "type": "User-only",
    "input": "User: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Natural Conversation",
    "input": "System: <text>\nUser: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Abstracted Context",
    "input": "[Tag]\nUser: <text>",
    "label": "{label}"
  }}
]
""".strip()


In [17]:
# ============================================================
# CELL 6 — OUTPUT CLEANING + CLAUDE CALL
# ============================================================
def clean_json_output(raw_output):
    text = raw_output.strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()
    return text


def extract_user_text(input_text):
    if not isinstance(input_text, str):
        return ""
    match = re.search(r"User:\s*(.*)", input_text, flags=re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else input_text.strip()


def call_claude_for_variations(system_text, user_text, label):
    prompt = build_augmentation_prompt(system_text, user_text, label)
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}],
    )
    cleaned = clean_json_output(response.content[0].text)
    parsed = json.loads(cleaned)
    if not isinstance(parsed, list):
        raise ValueError("Claude response is not a JSON list")
    return parsed


In [18]:
# ============================================================
# CELL 7 — SHARED GENERATION HELPER
# ============================================================
def generate_for_label(label, target, max_api_calls, seen_texts):
    label_sources = source_df[source_df["label"] == label].reset_index(drop=True)
    assert len(label_sources) > 0

    rows = []
    source_index = 0
    api_calls = 0

    print(
        f"\nStarting {label}: target={target}, "
        f"source_turns={len(label_sources)}, "
        f"source_conversations={label_sources['conversation_id'].nunique()}"
    )

    while len(rows) < target and api_calls < max_api_calls:
        source_row = label_sources.iloc[source_index % len(label_sources)]
        source_index += 1
        api_calls += 1

        try:
            variations = call_claude_for_variations(
                system_text=source_row["system_text"],
                user_text=source_row["user_text"],
                label=label,
            )

            for item in variations:
                if len(rows) >= target:
                    break
                if not isinstance(item, dict) or item.get("label") != label:
                    continue

                new_user_text = extract_user_text(item.get("input", ""))
                if not new_user_text:
                    continue

                normalized_new = re.sub(r"\s+", " ", new_user_text.strip().lower())
                normalized_original = re.sub(
                    r"\s+", " ", str(source_row["user_text"]).strip().lower()
                )

                # Reject unchanged source copy.
                if normalized_new == normalized_original:
                    continue

                # Reject exact duplicate generated user text globally for this fold.
                global_key = (label, normalized_new)
                if global_key in seen_texts:
                    continue
                seen_texts.add(global_key)

                rows.append({
                    "conversation_id": str(source_row["conversation_id"]),
                    "turn_index": int(source_row["turn_index"]),
                    "role": "user",
                    "text": new_user_text,
                    "intent": label,
                    "is_augmented": True,
                    "augmentation_type": item.get("type", ""),
                    "source_user_text": source_row["user_text"],
                    "source_system_text": source_row["system_text"],
                })

            if api_calls == 1 or api_calls % 10 == 0 or len(rows) >= target:
                print(f"{label}: {len(rows)}/{target} | API calls={api_calls}")

            time.sleep(0.5)

        except Exception as e:
            print(f"Error | CV{CV_RUN} | {label} | API call {api_calls}: {e}")
            time.sleep(1)

    print(f"Finished {label}: {len(rows)}/{target} | API calls={api_calls}")
    return rows


In [19]:
# ============================================================
# CELL 8A — GENERATE 100 CHITCHAT + 100 REVISE
# ============================================================
seen_texts = set()
core_rows = []

for label, target in CORE_TARGETS.items():
    rows = generate_for_label(
        label=label,
        target=target,
        max_api_calls=CORE_MAX_API_CALLS_PER_CLASS,
        seen_texts=seen_texts,
    )
    core_rows.extend(rows)

core_df = pd.DataFrame(core_rows)
print("\nCore synthetic distribution:")
print(core_df["intent"].value_counts())

for label, target in CORE_TARGETS.items():
    actual = int((core_df["intent"] == label).sum())
    assert actual == target, f"{label} stopped at {actual}/{target}"

CORE_OUTPUT_PATH = f"{OUTPUT_DIR}/cv{CV_RUN}_synthetic_100_chitchat_100_revise.csv"
core_df.to_csv(CORE_OUTPUT_PATH, index=False)
print("Saved core augmentation:", CORE_OUTPUT_PATH)



Starting Chitchat: target=100, source_turns=173, source_conversations=132
Chitchat: 1/100 | API calls=1
Chitchat: 15/100 | API calls=10
Chitchat: 15/100 | API calls=20
Chitchat: 29/100 | API calls=30
Chitchat: 42/100 | API calls=40
Chitchat: 56/100 | API calls=50
Chitchat: 68/100 | API calls=60
Chitchat: 81/100 | API calls=70
Error | CV1 | Chitchat | API call 71: Expecting value: line 1 column 1 (char 0)
Chitchat: 96/100 | API calls=80
Chitchat: 100/100 | API calls=82
Finished Chitchat: 100/100 | API calls=82

Starting Revise: target=100, source_turns=39, source_conversations=31
Revise: 3/100 | API calls=1
Revise: 23/100 | API calls=10
Revise: 44/100 | API calls=20
Revise: 64/100 | API calls=30
Revise: 85/100 | API calls=40
Revise: 97/100 | API calls=50
Revise: 100/100 | API calls=52
Finished Revise: 100/100 | API calls=52

Core synthetic distribution:
intent
Chitchat    100
Revise      100
Name: count, dtype: int64
Saved core augmentation: /content/drive/MyDrive/5class_5cv_augmentati